# Tratamento Inicial dos Datasets (com Indicadores Agro)

Este notebook aplica o tratamento inicial dos datasets base já com os indicadores agro necessários para os experimentos.

Fluxo esperado de execução:
1. `indicadores_agro/tratar_indicadores_agro.ipynb`
2. `tratamento_datasets.ipynb`
3. `variacoes_datasets.ipynb`

Processos realizados:
- Integra os lags de fechamento dos indicadores agro (câmbio e soja)
- Cria janelas futuras (3, 7, 15, 30 dias) para o preço de fechamento
- Gera rótulos binários (alta/baixa) para cada horizonte
- Remove linhas com NaN geradas pelos shifts
- Salva dois conjuntos tratados em `datasets_base`:
  - Regressão: valores originais + agro lags + fechamentos futuros
  - Classificação: valores originais + agro lags + targets futuros

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

print('Bibliotecas carregadas com sucesso.')

## 1. Configuração de pastas

In [ ]:
BASE_DIR = Path('..').resolve()  # Base/
DATASETS_DIR = BASE_DIR / 'datasets'
INPUT_AGRO_DIR = DATASETS_DIR / 'indicadores_agro' / 'indicador_agro_tratado'

OUTPUT_BASE_DIR = DATASETS_DIR / 'datasets_base'
OUTPUT_REGRESSAO_DIR = OUTPUT_BASE_DIR / 'regressao'
OUTPUT_CLASSIFICACAO_DIR = OUTPUT_BASE_DIR / 'classificacao'

OUTPUT_REGRESSAO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CLASSIFICACAO_DIR.mkdir(parents=True, exist_ok=True)

print('Pasta de entrada:', DATASETS_DIR)
print('Pasta de indicadores agro tratados:', INPUT_AGRO_DIR)
print('Pasta de saída (regressão):', OUTPUT_REGRESSAO_DIR)
print('Pasta de saída (classificação):', OUTPUT_CLASSIFICACAO_DIR)

## 2. Função de tratamento

In [ ]:
def _carregar_indicador_lags_fechamento(csv_path: Path, prefixo: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    coluna_data = 'data' if 'data' in df.columns else df.columns[0]
    df[coluna_data] = pd.to_datetime(df[coluna_data], errors='coerce')
    df = df.dropna(subset=[coluna_data]).set_index(coluna_data).sort_index()

    lag_cols = [c for c in df.columns if c.startswith('close_lag_')]
    if not lag_cols:
        raise ValueError(f'Nenhuma coluna close_lag_ encontrada em {csv_path.name}')

    # Apenas lags de fechamento dos indicadores agro entram nos experimentos.
    df = df[lag_cols].rename(columns={c: f'{prefixo}_{c}' for c in lag_cols})
    return df


def carregar_indicadores_agro_lags(input_agro_dir: Path) -> pd.DataFrame:
    soja = _carregar_indicador_lags_fechamento(
        input_agro_dir / 'Soja_Chicago_USD_Bushel.csv',
        'agro_soja'
    )
    cambio = _carregar_indicador_lags_fechamento(
        input_agro_dir / 'Cambio_USD_BRL.csv',
        'agro_cambio'
    )

    # Mantem apenas datas comuns entre indicadores para garantir consistencia.
    return soja.join(cambio, how='inner').sort_index()


def integrar_indicadores_agro_lags(df: pd.DataFrame, df_agro_lags: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date']).set_index('Date')
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index, errors='coerce')
        df = df[df.index.isna()]

    df = df.sort_index()
    df_integrado = df.join(df_agro_lags, how='inner').dropna(how='any')
    return df_integrado


def tratar_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
    elif isinstance(df.index, pd.DatetimeIndex):
        df = df.sort_index().reset_index().rename(columns={'index': 'Date'})
        if 'Date' not in df.columns:
            df = df.rename(columns={df.columns[0]: 'Date'})
    else:
        raise ValueError("A coluna 'Date' (ou índice datetime) é obrigatória para aplicar os shifts temporais com segurança.")

    # Criação de janelas futuras para o preço de fechamento
    df['Close_3d_fut'] = df['Close'].shift(-3)
    df['Close_7d_fut'] = df['Close'].shift(-7)
    df['Close_15d_fut'] = df['Close'].shift(-15)
    df['Close_30d_fut'] = df['Close'].shift(-30)

    # Remove linhas com NaN nas janelas futuras
    df = df.dropna(subset=['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut'])

    # Define índice temporal
    df = df.set_index('Date')

    # Cria rótulos binários (1 = alta, 0 = baixa/estável)
    df['target_3d'] = (df['Close_3d_fut'] > df['Close']).astype(int)
    df['target_7d'] = (df['Close_7d_fut'] > df['Close']).astype(int)
    df['target_15d'] = (df['Close_15d_fut'] > df['Close']).astype(int)
    df['target_30d'] = (df['Close_30d_fut'] > df['Close']).astype(int)

    future_cols = ['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut']
    target_cols = ['target_3d', 'target_7d', 'target_15d', 'target_30d']

    # Regressão: mantém valores originais + fechamentos futuros
    df_regressao = df.drop(columns=target_cols)

    # Classificação: mantém valores originais + targets (remove fechamentos futuros)
    df_classificacao = df.drop(columns=future_cols)

    return df_regressao, df_classificacao

## 3. Processamento em lote

In [ ]:
csv_files = sorted(DATASETS_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum CSV encontrado em {DATASETS_DIR}')

print(f'Arquivos encontrados: {len(csv_files)}')
for f in csv_files:
    print('-', f.name)

In [ ]:
agro_soja_path = INPUT_AGRO_DIR / 'Soja_Chicago_USD_Bushel.csv'
agro_cambio_path = INPUT_AGRO_DIR / 'Cambio_USD_BRL.csv'
if not agro_soja_path.exists() or not agro_cambio_path.exists():
    raise FileNotFoundError(
        'Indicadores agro tratados não encontrados! Execute primeiro indicadores_agro/tratar_indicadores_agro.ipynb'
    )

df_agro_lags = carregar_indicadores_agro_lags(INPUT_AGRO_DIR)
print('Indicadores agro (lags de fechamento) carregados:', df_agro_lags.shape)
print('Colunas agro utilizadas:', list(df_agro_lags.columns))

for file_path in csv_files:
    print('' + '=' * 80)
    print(f'Processando: {file_path.name}')

    df = pd.read_csv(file_path)
    print('Shape original:', df.shape)

    # Validação mínima
    required_cols = {'Date', 'Close'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        print(f'ERRO: colunas ausentes {missing_cols}. Arquivo ignorado.')
        continue

    df_com_agro = integrar_indicadores_agro_lags(df, df_agro_lags)
    print('Shape com agro lags:', df_com_agro.shape)

    df_regressao, df_classificacao = tratar_dataset(df_com_agro)
    print('Shape tratado (regressão):', df_regressao.shape)
    print('Shape tratado (classificação):', df_classificacao.shape)

    # Diagnóstico de desbalanceamento por horizonte
    targets = {
        '3d': df_classificacao['target_3d'],
        '7d': df_classificacao['target_7d'],
        '15d': df_classificacao['target_15d'],
        '30d': df_classificacao['target_30d'],
    }

    print('\n' + '=' * 70)
    print('DIAGNOSTICO DE DESBALANCEAMENTO POR HORIZONTE')
    print('=' * 70)

    for period, y in targets.items():
        y_clean = y.dropna()
        dist = y_clean.value_counts().sort_index()

        n0 = int(dist.get(0, 0))
        n1 = int(dist.get(1, 0))
        total = n0 + n1

        if total == 0:
            print(f'\n{period.upper()}: sem dados')
            continue

        p0 = n0 / total
        p1 = n1 / total

        major = max(n0, n1)
        minor = min(n0, n1)

        if minor == 0:
            imbalance_ratio = np.inf
            ratio_txt = 'infinito (apenas uma classe)'
        else:
            imbalance_ratio = major / minor
            ratio_txt = f'{imbalance_ratio:.2f}:1'

        # Baseline ingênua: sempre prever classe majoritária
        baseline_acc = major / total

        print(f'\nHorizonte {period.upper()}')
        print(f'  Classe 0 (Baixa/Estavel): {n0} ({p0:.1%})')
        print(f'  Classe 1 (Alta):          {n1} ({p1:.1%})')
        print(f'  Razao majoritaria/minoritaria: {ratio_txt}')
        print(f'  Baseline ingenua (sempre majoritaria): Accuracy = {baseline_acc:.4f}')

        if minor == 0:
            print('  ALERTA: target invalido para classificacao (so uma classe).')
        elif imbalance_ratio >= 3:
            print('  ALERTA: desbalanceamento relevante (>= 3:1).')
        else:
            print('  OK: distribuicao relativamente equilibrada (< 3:1).')

    # Salvar arquivos tratados
    output_name = file_path.stem + '_tratado.csv'
    output_path_reg = OUTPUT_REGRESSAO_DIR / output_name
    output_path_clf = OUTPUT_CLASSIFICACAO_DIR / output_name

    df_regressao.to_csv(output_path_reg, index=True)
    df_classificacao.to_csv(output_path_clf, index=True)

    print('Salvo (regressão) em:', output_path_reg)
    print('Salvo (classificação) em:', output_path_clf)

## 4. Resumo final

In [ ]:
processed_reg = sorted(OUTPUT_REGRESSAO_DIR.glob('*_tratado.csv'))
processed_clf = sorted(OUTPUT_CLASSIFICACAO_DIR.glob('*_tratado.csv'))

print('Arquivos tratados gerados (regressão):')
for f in processed_reg:
    print('-', f.name)

print('\nArquivos tratados gerados (classificação):')
for f in processed_clf:
    print('-', f.name)